
# CS2 EXP-3 — NeoBERT Frozen-Encoder Linear Probe (Nested CV + Canonical Retrain)



## 1. Runtime settings

In [1]:

from pathlib import Path

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "prashant"
REPO_ROOT = Path("/content/DiverseVul--IS-Project")
PROJECT_DIR = REPO_ROOT / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

DRIVE_ROOT = Path("/content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData")
PROCESSED_DIR = DRIVE_ROOT / "processed"
MANIFEST_ROOT = DRIVE_ROOT / "manifests"
OUTPUT_ROOT = DRIVE_ROOT / "outputs"

# Same frozen split used by CS1 EXP-0/1/2 -- CS2 reuses the identical
# project-grouped outer holdout + inner CV manifests so results are comparable.
SPLIT_ID = "cs1_project_holdout20_innercv_v1"

NORMALIZED_PARQUET = PROCESSED_DIR / "rdiversevul_cs1_normalized_v1.parquet"

OUTER_MANIFEST_PATH = MANIFEST_ROOT / SPLIT_ID / "outer_holdout" / "cs1_outer_project_holdout_manifest.parquet"
INNER_MANIFEST_PATH = MANIFEST_ROOT / SPLIT_ID / "inner_cv" / "cs1_project_grouped_5fold_manifest.parquet"

EXP3_OUTPUT_DIR = OUTPUT_ROOT / "case_study_2" / "exp3_linear_probe_nested_v1"
EMBEDDING_CACHE_DIR = EXP3_OUTPUT_DIR / "embedding_cache"

HF_CACHE_DIR = "/content/hf_cache"

RUN_PROFILE = False      # optional: inner C-search diagnostics for one outer fold
RUN_NESTED_OFFICIAL = True
RUN_CANONICAL_RETRAIN = True
RUN_HOLDOUT_EVAL = True

C_GRID = (1e-3, 1e-2, 1e-1, 1.0, 10.0)

print("Settings loaded.")
print("Normalized parquet:", NORMALIZED_PARQUET)
print("Output dir:", EXP3_OUTPUT_DIR)


Settings loaded.
Normalized parquet: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_v1.parquet
Output dir: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/outputs/case_study_2/exp3_linear_probe_nested_v1


## 2. Mount Google Drive and clone/refresh repository

In [3]:

from google.colab import drive
import os
import subprocess
import sys


def run_command(command, cwd=None):
    print("$", " ".join(str(x) for x in command))
    subprocess.run(command, check=True, cwd=cwd)


drive.mount("/content/drive")

if not REPO_ROOT.exists():
    run_command([
        "git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)
    ])
else:
    run_command(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH])
    run_command(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH])
    run_command(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH])

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f"Missing project directory: {PROJECT_DIR}")
if not SRC_DIR.is_dir():
    raise FileNotFoundError(f"Missing source directory: {SRC_DIR}")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("\nRepository ready.")
run_command(["git", "-C", str(REPO_ROOT), "log", "-1", "--oneline"])


Mounted at /content/drive
$ git clone --branch prashant --single-branch https://github.com/EnomisLP/DiverseVul--IS-Project.git /content/DiverseVul--IS-Project

Repository ready.
$ git -C /content/DiverseVul--IS-Project log -1 --oneline


## 3. Verify GPU runtime (embedding extraction is GPU-only)

In [4]:

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "EXP-3 requires a GPU runtime (Runtime > Change runtime type > GPU). "
        "NeoBERT embedding extraction is not supported on CPU in this pipeline."
    )

DEVICE = "cuda"
print("CUDA device:", torch.cuda.get_device_name(0))
print("bfloat16 supported:", torch.cuda.is_bf16_supported())


CUDA device: Tesla T4
bfloat16 supported: True


## 4. Install/import dependencies and verify committed files

In [5]:

import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers", "torch", "numpy", "pandas", "scikit-learn", "matplotlib",
    "pyarrow", "joblib",
], check=True)
print("Dependencies installed/verified.")


Dependencies installed/verified.


In [6]:

required_repo_files = [
    SRC_DIR / "case_study_1" / "split_manifest.py",
    SRC_DIR / "case_study_1" / "evaluation.py",
    SRC_DIR / "case_study_1" / "confidence_intervals.py",
    SRC_DIR / "case_study_2" / "data_loader.py",
    SRC_DIR / "case_study_2" / "models.py",
    SRC_DIR / "case_study_2" / "exp3" / "exp3_linear_probe.py",
]

missing = [str(p) for p in required_repo_files if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing required committed files:\n" + "\n".join(missing))

from case_study_1 import split_manifest
from case_study_1 import evaluation
from case_study_1 import confidence_intervals

from case_study_2.data_loader import create_dataloader
from case_study_2.models import configure_huggingface_cache, load_neobert_tokenizer, load_neobert_encoder
from case_study_2.exp3.exp3_linear_probe import (
    Exp3Config,
    NestedProbeConfig,
    extract_embeddings,
    run_exp3_nested_inner_profile,
    run_exp3_nested_probe,
    run_exp3_canonical_retrain,
    run_exp3_holdout_evaluation,
)

print("Imported project modules successfully.")


Imported project modules successfully.


## 5. Load dataset and frozen manifests

In [7]:

import pandas as pd

if not NORMALIZED_PARQUET.is_file():
    raise FileNotFoundError(f"Missing normalized dataset: {NORMALIZED_PARQUET}")
if not OUTER_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Missing outer manifest: {OUTER_MANIFEST_PATH}")
if not INNER_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Missing inner manifest: {INNER_MANIFEST_PATH}")

full_df = pd.read_parquet(NORMALIZED_PARQUET)
outer_manifest_df = pd.read_parquet(OUTER_MANIFEST_PATH)
inner_manifest_df = split_manifest.load_manifest(
    INNER_MANIFEST_PATH,
    config=split_manifest.SplitConfig(n_splits=5, random_state=42, shuffle=True),
)

required_columns = {"source_row_id", "normalized_code", "label", "project"}
missing_columns = required_columns.difference(full_df.columns)
if missing_columns:
    raise KeyError(f"Dataset missing required columns: {sorted(missing_columns)}")

print("Full dataset:", full_df.shape)
print("Outer manifest:", outer_manifest_df.shape)
print("Inner manifest:", inner_manifest_df.shape)
print(outer_manifest_df["partition"].value_counts(dropna=False))


Full dataset: (261667, 5)
Outer manifest: (261667, 5)
Inner manifest: (203958, 4)
partition
development      203958
outer_holdout     57709
Name: count, dtype: int64


## 6. Strict manifest and development-only validation

In [8]:

for name, frame in [("full_df", full_df), ("outer_manifest_df", outer_manifest_df), ("inner_manifest_df", inner_manifest_df)]:
    if frame["source_row_id"].duplicated().any():
        raise RuntimeError(f"{name} contains duplicate source_row_id values.")

full_indexed = full_df.set_index("source_row_id", drop=False)
dev_ids = set(outer_manifest_df.loc[outer_manifest_df["partition"] == "development", "source_row_id"].tolist())
holdout_ids = set(outer_manifest_df.loc[outer_manifest_df["partition"] == "outer_holdout", "source_row_id"].tolist())
inner_ids = set(inner_manifest_df["source_row_id"].tolist())

if dev_ids.intersection(holdout_ids):
    raise RuntimeError("Development and outer holdout ID overlap detected.")
if inner_ids != dev_ids:
    raise RuntimeError(
        f"Inner CV manifest IDs must exactly equal development IDs. "
        f"Missing={len(dev_ids - inner_ids)}, extra={len(inner_ids - dev_ids)}"
    )

dev_projects = set(outer_manifest_df.loc[outer_manifest_df["partition"] == "development", "project"].astype(str))
holdout_projects = set(outer_manifest_df.loc[outer_manifest_df["partition"] == "outer_holdout", "project"].astype(str))
if dev_projects.intersection(holdout_projects):
    raise RuntimeError("Development/holdout project overlap detected.")

development_frame = full_indexed.loc[list(dev_ids)].copy().reset_index(drop=True)
holdout_frame = full_indexed.loc[list(holdout_ids)].copy().reset_index(drop=True)

for frame in (development_frame, holdout_frame):
    frame["normalized_code"] = frame["normalized_code"].fillna("").astype(str)
    empty_mask = frame["normalized_code"].str.strip().eq("")
    if empty_mask.any():
        frame.loc[empty_mask, "normalized_code"] = "EMPTY_CODE_SAMPLE"

print("Development rows:", len(development_frame), "| projects:", development_frame["project"].nunique())
print("Outer holdout rows:", len(holdout_frame), "| projects:", holdout_frame["project"].nunique())
print("Leakage check passed: development/outer holdout are disjoint by row and by project.")


Development rows: 203958 | projects: 594
Outer holdout rows: 57709 | projects: 203
Leakage check passed: development/outer holdout are disjoint by row and by project.


## 7. Configure EXP-3 linear probe

In [9]:

base_config = Exp3Config(
    experiment_name="cs2_exp3_neobert_linear_probe",
    code_column="normalized_code",
    source_id_column="source_row_id",
    label_column="label",
    project_column="project",
    fold_column="fold",
    hf_cache_dir=HF_CACHE_DIR,
    max_length=512,
    dtype_policy="bfloat16",
    embedding_batch_size=32,
    logistic_max_iter=2000,
    logistic_solver="lbfgs",
    class_weight="balanced",
    decision_threshold=0.50,
    random_state=42,
    verbose=True,
)

nested_config = NestedProbeConfig(
    experiment_name="cs2_exp3_nested_probe_dev_grouped",
    C_grid=C_GRID,
    inner_n_splits=3,
    inner_random_state=20260707,
    selection_metric="average_precision_pr_auc",
    decision_threshold=0.50,
    tie_break_rule="higher_C_then_grid_order",
    verbose=True,
)

print("Model:", base_config.model_name, "| Tokenizer:", base_config.tokenizer_name)
print("C grid:", nested_config.C_grid)
print("Output directory:", EXP3_OUTPUT_DIR)


Model: chandar-lab/NeoBERT | Tokenizer: google-bert/bert-base-uncased
C grid: (0.001, 0.01, 0.1, 1.0, 10.0)
Output directory: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/outputs/case_study_2/exp3_linear_probe_nested_v1


## 8. Load frozen NeoBERT encoder and extract embeddings (GPU, once)

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"  

from google.colab import drive
drive.mount('/content/drive', force_remount=False)


EMBEDDING_CACHE_DIR = Path("/content/drive/MyDrive/case_study_2/embedding_cache")
EMBEDDING_CACHE_DIR.mkdir(parents=True, exist_ok=True)

configure_huggingface_cache(HF_CACHE_DIR)
tokenizer = load_neobert_tokenizer(base_config.tokenizer_name, hf_cache_dir=HF_CACHE_DIR)

print(f"Loading frozen NeoBERT encoder on {DEVICE}; dtype_policy={base_config.dtype_policy}")
encoder = load_neobert_encoder(
    base_config.model_name,
    dtype_policy=base_config.dtype_policy,
    device=DEVICE,
    freeze=True,
    hf_cache_dir=HF_CACHE_DIR,
)

base_config.embedding_batch_size = 64

print(f"[monitor] development_frame: {len(development_frame)} righe")
print(f"[monitor] holdout_frame: {len(holdout_frame)} righe")

print("\n=== Estrazione embedding: development ===")
development_embeddings = extract_embeddings(
    encoder, tokenizer, development_frame, base_config, DEVICE,
    cache_path=EMBEDDING_CACHE_DIR / "development_embeddings.npy",
)

print("\n=== Estrazione embedding: holdout ===")
holdout_embeddings = extract_embeddings(
    encoder, tokenizer, holdout_frame, base_config, DEVICE,
    cache_path=EMBEDDING_CACHE_DIR / "holdout_embeddings.npy",
)

print("\nDevelopment embeddings:", development_embeddings.shape)
print("Holdout embeddings:", holdout_embeddings.shape)

del encoder
torch.cuda.empty_cache()

In [ ]:
# ============================================================
# 8. Load frozen NeoBERT encoder and extract embeddings
#    with progress tracking and cache
# ============================================================

import math
import time
import numpy as np
import torch
from tqdm.auto import tqdm

configure_huggingface_cache(HF_CACHE_DIR)

tokenizer = load_neobert_tokenizer(
    base_config.tokenizer_name,
    hf_cache_dir=HF_CACHE_DIR,
)

print(f"Loading frozen NeoBERT encoder on {DEVICE}; dtype_policy={base_config.dtype_policy}")

encoder = load_neobert_encoder(
    base_config.model_name,
    dtype_policy=base_config.dtype_policy,
    device=DEVICE,
    freeze=True,
    hf_cache_dir=HF_CACHE_DIR,
)

EMBEDDING_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def extract_embeddings_with_progress(
    encoder,
    tokenizer,
    frame,
    config,
    device,
    cache_path,
    name,
):
    cache_path = Path(cache_path)

    if cache_path.exists():
        print(f"[{name}] Loading cached embeddings:")
        print(cache_path)
        arr = np.load(cache_path)
        print(f"[{name}] Cached shape:", arr.shape)
        return arr

    print(f"\n[{name}] Extracting embeddings")
    print(f"[{name}] rows:", len(frame))
    print(f"[{name}] batch_size:", config.embedding_batch_size)
    print(f"[{name}] max_length:", config.max_length)
    print(f"[{name}] cache_path:", cache_path)

    dataloader = create_dataloader(
        frame,
        tokenizer,
        batch_size=config.embedding_batch_size,
        max_length=config.max_length,
        shuffle=False,
        code_column=config.code_column,
    )

    total_batches = math.ceil(len(frame) / config.embedding_batch_size)
    chunks = []

    encoder.eval()
    start_time = time.time()

    with torch.inference_mode():
        for batch_idx, batch in enumerate(
            tqdm(dataloader, total=total_batches, desc=f"{name} embeddings")
        ):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

            # CLS embedding
            pooled = outputs.last_hidden_state[:, 0, :]

            chunks.append(pooled.float().cpu().numpy())

            if (batch_idx + 1) % 100 == 0:
                elapsed_min = (time.time() - start_time) / 60
                done_rows = min((batch_idx + 1) * config.embedding_batch_size, len(frame))
                print(
                    f"[{name}] {done_rows}/{len(frame)} rows "
                    f"({100 * done_rows / len(frame):.2f}%) "
                    f"elapsed={elapsed_min:.1f} min"
                )

    embeddings = np.concatenate(chunks, axis=0)

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(cache_path, embeddings)

    elapsed_min = (time.time() - start_time) / 60
    print(f"[{name}] Done. Shape={embeddings.shape}. Time={elapsed_min:.2f} min")
    print(f"[{name}] Saved cache:", cache_path)

    return embeddings


development_embeddings = extract_embeddings_with_progress(
    encoder=encoder,
    tokenizer=tokenizer,
    frame=development_frame,
    config=base_config,
    device=DEVICE,
    cache_path=EMBEDDING_CACHE_DIR / "development_embeddings.npy",
    name="development",
)

holdout_embeddings = extract_embeddings_with_progress(
    encoder=encoder,
    tokenizer=tokenizer,
    frame=holdout_frame,
    config=base_config,
    device=DEVICE,
    cache_path=EMBEDDING_CACHE_DIR / "holdout_embeddings.npy",
    name="holdout",
)

print("Development embeddings:", development_embeddings.shape)
print("Holdout embeddings:", holdout_embeddings.shape)

del encoder
torch.cuda.empty_cache()

Loading frozen NeoBERT encoder on cuda; dtype_policy=bfloat16


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] NeoBERT LOAD REPORT from: chandar-lab/NeoBERT
Key            | Status     |  | 
---------------+------------+--+-
decoder.weight | UNEXPECTED |  | 
decoder.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[development] Extracting embeddings
[development] rows: 203958
[development] batch_size: 32
[development] max_length: 512
[development] cache_path: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/outputs/case_study_2/exp3_linear_probe_nested_v1/embedding_cache/development_embeddings.npy


development embeddings:   0%|          | 0/6374 [00:00<?, ?it/s]

[development] 3200/203958 rows (1.57%) elapsed=6.2 min
[development] 6400/203958 rows (3.14%) elapsed=12.4 min
[development] 9600/203958 rows (4.71%) elapsed=18.6 min
[development] 12800/203958 rows (6.28%) elapsed=24.8 min
[development] 16000/203958 rows (7.84%) elapsed=30.9 min
[development] 19200/203958 rows (9.41%) elapsed=37.1 min
[development] 22400/203958 rows (10.98%) elapsed=43.2 min
[development] 25600/203958 rows (12.55%) elapsed=49.4 min
[development] 28800/203958 rows (14.12%) elapsed=55.6 min
[development] 32000/203958 rows (15.69%) elapsed=61.8 min
[development] 35200/203958 rows (17.26%) elapsed=68.0 min
[development] 38400/203958 rows (18.83%) elapsed=74.3 min
[development] 41600/203958 rows (20.40%) elapsed=80.5 min
[development] 44800/203958 rows (21.97%) elapsed=86.7 min
[development] 48000/203958 rows (23.53%) elapsed=92.9 min
[development] 51200/203958 rows (25.10%) elapsed=99.1 min
[development] 54400/203958 rows (26.67%) elapsed=105.2 min
[development] 57600/203

## 9. Optional profile run — inner C-search only, one outer fold

In [ ]:

if RUN_PROFILE:
    profile = run_exp3_nested_inner_profile(
        development_frame=development_frame,
        development_embeddings=development_embeddings,
        development_manifest=inner_manifest_df,
        outer_fold_id=4,
        base_config=base_config,
        nested_config=nested_config,
    )
    print("Profile duration minutes:", profile["total_profile_seconds"] / 60)
    print("Selected C:")
    display(pd.DataFrame([profile["selected_C"]]))
    print("Inner C summary:")
    display(profile["C_summary"])
else:
    print("RUN_PROFILE=False; skipping profile.")


## 10. Official nested cross-validation (development-only, pooled OOF)

In [ ]:

if RUN_NESTED_OFFICIAL:
    results = run_exp3_nested_probe(
        development_frame=development_frame,
        development_embeddings=development_embeddings,
        development_manifest=inner_manifest_df,
        base_config=base_config,
        nested_config=nested_config,
        output_dir=EXP3_OUTPUT_DIR,
        additional_metadata={
            "run_kind": "exp3_linear_probe_nested_cv_development_only",
            "global_outer_holdout_used": False,
            "input_parquet": str(NORMALIZED_PARQUET),
            "input_column": base_config.code_column,
            "outer_manifest_path": str(OUTER_MANIFEST_PATH),
            "inner_manifest_path": str(INNER_MANIFEST_PATH),
            "repo_commit": subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"]).decode().strip(),
        },
    )
    print("\nOfficial EXP-3 nested run complete.")
else:
    results = None
    print("RUN_NESTED_OFFICIAL=False; official nested run skipped.")


## 11. Display nested result summary

In [ ]:

if results is None:
    print("No results object in memory. Set RUN_NESTED_OFFICIAL=True.")
else:
    pooled = results["evaluation"]["pooled_metrics"]
    fold_metrics = results["evaluation"]["fold_metrics"]
    selected = results["selected_alpha"]
    outer_training = results["outer_fold_training"]

    print("Pooled nested OOF metrics:")
    display(pd.DataFrame([{"metric": k, "value": v} for k, v in pooled.items()]))

    print("Selected C by frozen outer-development fold:")
    display(selected)

    print("Fold metrics:")
    display(fold_metrics)

    print("Outer fold training / leakage audit:")
    display(outer_training)

    print("Artifacts:")
    print(results["artifacts"])


## 11b. Confidence interval on development-CV pooled PR-AUC

In [ ]:

exp3_devcv_ci = confidence_intervals.bootstrap_metric_ci(
    results["oof_predictions"],
    metric="average_precision_pr_auc",
    n_bootstrap=1000,
    random_state=42,
)
print(confidence_intervals.format_ci_report(exp3_devcv_ci))


## 12. Compare against previous development-only reference results

In [ ]:

reference_rows = [
    {"experiment": "EXP-0 fixed normalized_code", "scope": "development pooled OOF", "pr_auc": 0.125205},
    {"experiment": "EXP-0 nested-alpha normalized_code", "scope": "development nested pooled OOF", "pr_auc": 0.141971},
    {"experiment": "EXP-2 MLP fixed representation", "scope": "development pooled OOF", "pr_auc": 0.138157},
]

if results is not None:
    reference_rows.append({
        "experiment": "EXP-3 NeoBERT linear probe (nested C)",
        "scope": "development nested pooled OOF",
        "pr_auc": float(results["evaluation"]["pooled_metrics"]["average_precision_pr_auc"]),
    })

comparison_df = pd.DataFrame(reference_rows).sort_values("pr_auc", ascending=False).reset_index(drop=True)
display(comparison_df)


## 13. Canonical retraining: fit final probe on the complete development set

In [ ]:

if RUN_CANONICAL_RETRAIN:
    if results is not None and len(results["selected_alpha"]):
        optimal_C = float(results["selected_alpha"]["selected_C"].mode()[0])
    else:
        optimal_C = 1.0

    print("Selected canonical C (modal value across outer folds):", optimal_C)

    final_scaler, final_clf = run_exp3_canonical_retrain(
        development_frame=development_frame,
        development_embeddings=development_embeddings,
        selected_C=optimal_C,
        base_config=base_config,
        output_dir=EXP3_OUTPUT_DIR,
    )
    print("Canonical probe saved to:", EXP3_OUTPUT_DIR)
else:
    final_scaler = final_clf = None
    print("RUN_CANONICAL_RETRAIN=False; skipping canonical retraining.")


## 14. Final evaluation: score frozen 20% outer holdout partition

In [ ]:

if RUN_HOLDOUT_EVAL and final_clf is not None:
    holdout_results = run_exp3_holdout_evaluation(
        holdout_frame=holdout_frame,
        holdout_embeddings=holdout_embeddings,
        scaler=final_scaler,
        clf=final_clf,
        base_config=base_config,
        output_dir=EXP3_OUTPUT_DIR,
    )
    holdout_predictions_df = holdout_results["holdout_predictions"]
    holdout_metrics = holdout_results["holdout_metrics"]
else:
    holdout_results = None
    print("RUN_HOLDOUT_EVAL=False or no canonical model available; skipping.")


## 14b. Confidence interval on outer-holdout PR-AUC

In [ ]:

exp3_holdout_ci = confidence_intervals.bootstrap_metric_ci(
    holdout_predictions_df,
    metric="average_precision_pr_auc",
    n_bootstrap=1000,
    random_state=42,
)
print(confidence_intervals.format_ci_report(exp3_holdout_ci))


## 15. Interpretability audit: top probe weight dimensions

In [ ]:

coef = final_clf.coef_[0]
weight_df = pd.DataFrame({
    "embedding_dim": np.arange(len(coef)),
    "weight": coef,
}).sort_values("weight", ascending=False).reset_index(drop=True)

print("Top 15 dimensions pushing toward Vulnerable:")
display(weight_df.head(15))
print("\nTop 15 dimensions pushing toward Non-Vulnerable:")
display(weight_df.tail(15))

weight_df.to_csv(EXP3_OUTPUT_DIR / "exp3_probe_weights.csv", index=False)


## 16. Error analysis: isolate holdout false positives & false negatives

In [ ]:

holdout_res = holdout_frame.copy()
holdout_res["y_score"] = holdout_predictions_df["y_score"].values
holdout_res["y_pred"] = holdout_results["y_pred"]

false_positives = holdout_res[(holdout_res["label"] == 0) & (holdout_res["y_pred"] == 1)]
false_negatives = holdout_res[(holdout_res["label"] == 1) & (holdout_res["y_pred"] == 0)]

print(f"Extracted {len(false_positives)} False Positives and {len(false_negatives)} False Negatives.")

false_positives[["source_row_id", "project", "y_score", "normalized_code"]].sample(
    n=min(5, len(false_positives)), random_state=42
).to_csv(EXP3_OUTPUT_DIR / "sample_false_positives.csv", index=False)

false_negatives[["source_row_id", "project", "y_score", "normalized_code"]].sample(
    n=min(5, len(false_negatives)), random_state=42
).to_csv(EXP3_OUTPUT_DIR / "sample_false_negatives.csv", index=False)
